In [1]:
import pandas as pd

In [2]:
!pip install datasets

In [3]:
!pip install transformers accelerate sentencepiece torch

In [4]:
!pip install --upgrade pip setuptools wheel

  Using cached setuptools-82.0.1-py3-none-any.whl.metadata (6.5 kB)
Using cached setuptools-82.0.1-py3-none-any.whl (1.0 MB)
  Attempting uninstall: setuptools
    Found existing installation: setuptools 81.0.0
    Uninstalling setuptools-81.0.0:
      Successfully uninstalled setuptools-81.0.0
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
torch 2.12.1 requires setuptools<82, but you have setuptools 82.0.1 which is incompatible.


In [5]:
!pip install datasets transformers accelerate

  Using cached setuptools-81.0.0-py3-none-any.whl.metadata (6.6 kB)
Using cached setuptools-81.0.0-py3-none-any.whl (1.1 MB)
  Attempting uninstall: setuptools
    Found existing installation: setuptools 82.0.1
    Uninstalling setuptools-82.0.1:
      Successfully uninstalled setuptools-82.0.1


In [6]:
pip list | grep datasets

datasets                5.0.0
Note: you may need to restart the kernel to use updated packages.


In [1]:
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    Trainer,
    TrainingArguments,
    DataCollatorWithPadding,
)

/home/kinkini/Desktop/multilingual-fake-news-detector/venv312/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
from datasets import Dataset
import pandas as pd
import torch

In [3]:
import pandas as pd
df = pd.read_csv(
    "/home/kinkini/Desktop/multilingual-fake-news-detector/data/processed_news.csv"
)

dataset = Dataset.from_pandas(df)

dataset = dataset.train_test_split(
    test_size=0.2,
    seed=42
)

In [4]:
MODEL_NAME = "xlm-roberta-base"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)


In [5]:
def tokenize(batch):
    return tokenizer(
        batch["text"],
        truncation=True,
        max_length=128      # reduced from 512
    )

dataset = dataset.map(
    tokenize,
    batched=True
)

dataset = dataset.remove_columns(["text"])

dataset.set_format(
    "torch"
)


Map: 100%|██████████| 8980/8980 [00:04<00:00, 2076.49 examples/s]


In [6]:
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=2
)

base_model = model.base_model

# Freeze embeddings
for param in base_model.embeddings.parameters():
    param.requires_grad = False

# Freeze first 8 encoder layers
for layer in base_model.encoder.layer[:8]:
    for param in layer.parameters():
        param.requires_grad = False


Loading weights: 100%|██████████| 197/197 [00:00<00:00, 1667.38it/s]
[transformers] XLMRobertaForSequenceClassification LOAD REPORT from: xlm-roberta-base
Key                         | Status     | 
----------------------------+------------+-
lm_head.dense.weight        | UNEXPECTED | 
roberta.pooler.dense.weight | UNEXPECTED | 
lm_head.bias                | UNEXPECTED | 
lm_head.dense.bias          | UNEXPECTED | 
lm_head.layer_norm.weight   | UNEXPECTED | 
roberta.pooler.dense.bias   | UNEXPECTED | 
lm_head.layer_norm.bias     | UNEXPECTED | 
classifier.dense.bias       | MISSING    | 
classifier.out_proj.bias    | MISSING    | 
classifier.out_proj.weight  | MISSING    | 
classifier.dense.weight     | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [7]:
training_args = TrainingArguments(
    output_dir="./xlmr_model",

    num_train_epochs=2,

    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,

    gradient_accumulation_steps=8,

    learning_rate=2e-5,

    fp16=torch.cuda.is_available(),

    eval_strategy="epoch",

    save_strategy="epoch",

    logging_steps=20,

    load_best_model_at_end=True,

    save_total_limit=1,

    report_to="none",

    dataloader_num_workers=0
)


In [16]:
from transformers import Trainer, DataCollatorWithPadding

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=dataset["train"],
    eval_dataset=dataset["test"],
    processing_class=tokenizer,
    data_collator=DataCollatorWithPadding(tokenizer),
)

In [24]:
# Optional: train on smaller data first
small_train = dataset["train"].shuffle(seed=42).select(range(5000))
small_test = dataset["test"].shuffle(seed=42).select(range(1000))

training_args = TrainingArguments(
    output_dir="./xlmr_model",

    num_train_epochs=1,              # was 2
    per_device_train_batch_size=8,   # was 2, increase if GPU allows
    per_device_eval_batch_size=8,

    gradient_accumulation_steps=2,   # effective batch = 16

    learning_rate=2e-5,
    fp16=torch.cuda.is_available(),

    eval_strategy="no",              # faster while testing
    save_strategy="no",              # don't save checkpoints each epoch
    logging_steps=100,

    report_to="none",
    dataloader_num_workers=2
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=small_train,       # use dataset["train"] for full training
    eval_dataset=small_test,
    processing_class=tokenizer,
    data_collator=DataCollatorWithPadding(tokenizer),
)

trainer.train()

trainer.save_model("models/xlmr")
tokenizer.save_pretrained("models/xlmr")

print("Model saved.")

/home/kinkini/Desktop/multilingual-fake-news-detector/venv312/lib/python3.12/site-packages/torch/utils/data/dataloader.py:1095: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Step,Training Loss
100,0.832016
200,0.143504
300,0.083389


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Model saved.
